In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from prophet import Prophet
from lightgbm import LGBMRegressor
from sklearn import metrics
from sklearn.model_selection import train_test_split, GridSearchCV , TimeSeriesSplit
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

In [ ]:
dir = Path('Data') 
 
files = [
    'customers', 'geography', 'inventory', 'order_items', 'orders',
    'payments', 'products', 'promotions', 'returns', 'reviews',
    'sales', 'shipments', 'web_traffic', 'sample_submission'
]

dfs = {}

for file in files:
    dfs[file] = pd.read_csv(dir / f'{file}.csv')



C:\Users\Admin\AppData\Local\Temp\ipykernel_27656\2233918903.py:12: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs[file] = pd.read_csv(dir / f'{file}.csv')


In [ ]:
test = dfs['sample_submission']

In [ ]:
df = dfs['sales']

df['Date'] = pd.to_datetime(df['Date'])

df['month'] = df['Date'].dt.month
df['day_of_month'] = df['Date'].dt.day
df['week_of_year'] = df['Date'].dt.isocalendar().week.astype(int)
df['is_weekend'] = (df['Date'].dt.dayofweek >= 5).astype(int)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
df['day_of_month_sin'] = np.sin(2 * np.pi * df['day_of_month'] / 31)
df['day_of_month_cos'] = np.cos(2 * np.pi * df['day_of_month'] / 31)

df['lag_7'] = df['Revenue'].shift(7)
df['lag_30'] = df['Revenue'].shift(30)

df['roll_mean_7'] = df['Revenue'].shift(1).rolling(7).mean()
df['roll_mean_30'] = df['Revenue'].shift(1).rolling(30).mean()
df['roll_std_7'] = df['Revenue'].shift(1).rolling(7).std()

df['growth_1'] = df['Revenue'].pct_change(1).shift(1)
df['growth_7'] = df['Revenue'].pct_change(7).shift(1)
df['growth_30'] = df['Revenue'].pct_change(30).shift(1)

df['cogs_lag_7'] = df['COGS'].shift(7)
df['cogs_lag_30'] = df['COGS'].shift(30)

df['cogs_roll_mean_7'] = df['COGS'].shift(1).rolling(7).mean()
df['cogs_roll_mean_30'] = df['COGS'].shift(1).rolling(30).mean()
df['cogs_roll_std_7'] = df['COGS'].shift(1).rolling(7).std()

df['cogs_growth_1'] = df['COGS'].pct_change(1).shift(1)
df['cogs_growth_7'] = df['COGS'].pct_change(7).shift(1)
df['cogs_growth_30'] = df['COGS'].pct_change(30).shift(1)

In [ ]:
x = df.drop(columns = ['Revenue', 'Date', 'COGS', 'cogs_lag_7','cogs_lag_30','cogs_roll_mean_7','cogs_roll_mean_30', 'cogs_growth_30', 'cogs_roll_std_7','cogs_growth_1', 'cogs_growth_7'  ])
y = df['Revenue']
x1 = df.drop(columns = ['Revenue', 'Date', 'COGS', 'lag_7','lag_30','roll_mean_7','roll_mean_30', 'roll_std_7','growth_1', 'growth_7', 'growth_30'])
y1 = df['COGS']

## 1. LightGBM

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

gbm = LGBMRegressor(
    objective='regression',
    random_state=42
)

param_grid_gbm = {
    'n_estimators': [50, 100,200],
    'max_depth': [ 5, 7],
    'learning_rate': [0.05, 0.1],
    'subsample': [ 0.8, 1.0],
    'colsample_bytree': [ 0.8, 1.0]
}

grid_gbm = GridSearchCV(
    estimator=gbm,
    param_grid=param_grid_gbm,
    cv=tscv,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=1
)



In [ ]:
grid_gbm.fit(x,y)
gbm_rev_be = grid_gbm.best_estimator_
grid_gbm.best_score_

Fitting 5 folds for each of 48 candidates, totalling 240 fits


In [ ]:
grid_gbm.fit(x1,y1)
gbm_cogs_be = grid_gbm.best_estimator_
grid_gbm.best_score_

Fitting 5 folds for each of 48 candidates, totalling 240 fits
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000219 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2215
[LightGBM] [Info] Number of data points in the train set: 3833, number of used features: 16
[LightGBM] [Info] Start training from score 3695134.492361
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best ga

np.float64(-655324.9817778534)

In [ ]:
test['Date'] = pd.to_datetime(test['Date'])

history = df[['Date', 'Revenue', 'COGS']].copy()
history = history.sort_values('Date').reset_index(drop=True)
def create_features(date, history):
    date = pd.to_datetime(date)

    rev = history['Revenue']
    cogs = history['COGS']

    row = {
        # time features
        'month': date.month,
        'day_of_month': date.day,
        'week_of_year': int(date.isocalendar().week),
        'is_weekend': int(date.dayofweek >= 5),
        'month_sin': np.sin(2 * np.pi * date.month / 12),
        'month_cos': np.cos(2 * np.pi * date.month / 12),
        'day_of_month_sin': np.sin(2 * np.pi * date.day / 31),
        'day_of_month_cos': np.cos(2 * np.pi * date.day / 31),

        # Revenue features
        'lag_7': rev.iloc[-7],
        'lag_30': rev.iloc[-30],
        'roll_mean_7': rev.iloc[-7:].mean(),
        'roll_mean_30': rev.iloc[-30:].mean(),
        'roll_std_7': rev.iloc[-7:].std(),
        'growth_1': rev.iloc[-1] / rev.iloc[-2] - 1,
        'growth_7': rev.iloc[-1] / rev.iloc[-8] - 1,
        'growth_30': rev.iloc[-1] / rev.iloc[-31] - 1,
       

        # COGS features
        'cogs_lag_7': cogs.iloc[-7],
        'cogs_lag_30': cogs.iloc[-30],
        'cogs_roll_mean_7': cogs.iloc[-7:].mean(),
        'cogs_roll_mean_30': cogs.iloc[-30:].mean(),
        'cogs_roll_std_7': cogs.iloc[-7:].std(),
        'cogs_growth_1': cogs.iloc[-1] / cogs.iloc[-2] - 1,
        'cogs_growth_7': cogs.iloc[-1] / cogs.iloc[-8] - 1,
        'cogs_growth_30': cogs.iloc[-1] / cogs.iloc[-31] - 1,

    }

    return pd.DataFrame([row])




In [ ]:
pred_revenue_gbm = []
pred_cogs_gbm = []

for date in test['Date']:

    X_input = create_features(date, history)

    X_rev_gbm = X_input.reindex(columns=x.columns, fill_value=0)
    X_cogs_gbm = X_input.reindex(columns=x1.columns, fill_value=0)

    rev_pred_gbm = gbm_rev_be.predict(X_rev_gbm)[0]
    cogs_pred_gbm = gbm_cogs_be.predict(X_cogs_gbm)[0]

    pred_revenue_gbm.append(rev_pred_gbm)
    pred_cogs_gbm.append(cogs_pred_gbm)

    history = pd.concat([
        history,
        pd.DataFrame({
            'Date': [date],
            'Revenue': [rev_pred_gbm],
            'COGS': [cogs_pred_gbm]
        })
    ], ignore_index=True)

## 2. XGBOOST


In [ ]:

tscv = TimeSeriesSplit(n_splits=5)

xgb = XGBRegressor(
    objective='reg:squarederror',
    random_state=42
)

param_grid_xgb = {
    'n_estimators': [50, 100,200],
    'max_depth': [ 5, 7],
    'learning_rate': [0.05, 0.1],
    'subsample': [ 0.8, 1.0],
    'colsample_bytree': [ 0.8, 1.0]
}

grid_xgb = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid_xgb,
    cv=tscv,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=1
)



In [ ]:
grid_xgb.fit(x,y)
xgb_rev_be = grid_xgb.best_estimator_
grid_xgb.best_score_

Fitting 5 folds for each of 48 candidates, totalling 240 fits


np.float64(-751225.9936238244)

In [ ]:
grid_xgb.fit(x1,y1)
xgb_cogs_be = grid_xgb.best_estimator_
grid_xgb.best_score_

Fitting 5 folds for each of 48 candidates, totalling 240 fits


np.float64(-651049.0958377743)

In [ ]:
test['Date'] = pd.to_datetime(test['Date'])

history = df[['Date', 'Revenue', 'COGS']].copy()
history = history.sort_values('Date').reset_index(drop=True)
def create_features(date, history):
    date = pd.to_datetime(date)

    rev = history['Revenue']
    cogs = history['COGS']

    row = {
        # time features
        'month': date.month,
        'day_of_month': date.day,
        'week_of_year': int(date.isocalendar().week),
        'is_weekend': int(date.dayofweek >= 5),
        'month_sin': np.sin(2 * np.pi * date.month / 12),
        'month_cos': np.cos(2 * np.pi * date.month / 12),
        'day_of_month_sin': np.sin(2 * np.pi * date.day / 31),
        'day_of_month_cos': np.cos(2 * np.pi * date.day / 31),

        # Revenue features
        'lag_7': rev.iloc[-7],
        'lag_30': rev.iloc[-30],
        'roll_mean_7': rev.iloc[-7:].mean(),
        'roll_mean_30': rev.iloc[-30:].mean(),
        'roll_std_7': rev.iloc[-7:].std(),
        'growth_1': rev.iloc[-1] / rev.iloc[-2] - 1,
        'growth_7': rev.iloc[-1] / rev.iloc[-8] - 1,
        'growth_30': rev.iloc[-1] / rev.iloc[-31] - 1,
       

        # COGS features
        'cogs_lag_7': cogs.iloc[-7],
        'cogs_lag_30': cogs.iloc[-30],
        'cogs_roll_mean_7': cogs.iloc[-7:].mean(),
        'cogs_roll_mean_30': cogs.iloc[-30:].mean(),
        'cogs_roll_std_7': cogs.iloc[-7:].std(),
        'cogs_growth_1': cogs.iloc[-1] / cogs.iloc[-2] - 1,
        'cogs_growth_7': cogs.iloc[-1] / cogs.iloc[-8] - 1,
        'cogs_growth_30': cogs.iloc[-1] / cogs.iloc[-31] - 1,

    }

    return pd.DataFrame([row])




In [ ]:
pred_revenue_xgb = []
pred_cogs_xgb = []

for date in test['Date']:

    X_input = create_features(date, history)

    # khớp đúng cột mà mỗi model đã được train
    X_rev_xgb = X_input.reindex(columns=x.columns, fill_value=0)
    X_cogs_xgb = X_input.reindex(columns=x1.columns, fill_value=0)

    rev_pred_xgb = xgb_rev_be.predict(X_rev_xgb)[0]
    cogs_pred_xgb = xgb_cogs_be.predict(X_cogs_xgb)[0]

    pred_revenue_xgb.append(rev_pred_xgb)
    pred_cogs_xgb.append(cogs_pred_xgb)

    history = pd.concat([
        history,
        pd.DataFrame({
            'Date': [date],
            'Revenue': [rev_pred_xgb],
            'COGS': [cogs_pred_xgb]
        })
    ], ignore_index=True)

## 3. Prophet

In [ ]:
revenue_df = df[['Date', 'Revenue']].copy()
revenue_df.columns = ['ds', 'y']
revenue_df['ds'] = pd.to_datetime(revenue_df['ds'])

cogs_df = df[['Date', 'COGS']].copy()
cogs_df.columns = ['ds', 'y']
cogs_df['ds'] = pd.to_datetime(cogs_df['ds'])



In [ ]:
prophet_revenue = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    seasonality_mode='multiplicative',
    interval_width=0.95,
    changepoint_prior_scale=0.05
)
prophet_revenue.fit(revenue_df)

13:18:09 - cmdstanpy - INFO - Chain [1] start processing
13:18:11 - cmdstanpy - INFO - Chain [1] done processing


In [ ]:
prophet_cogs = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    seasonality_mode='multiplicative',
    interval_width=0.95,
    changepoint_prior_scale=0.05
)
prophet_cogs.fit(cogs_df)

13:18:11 - cmdstanpy - INFO - Chain [1] start processing
13:18:11 - cmdstanpy - INFO - Chain [1] done processing


In [ ]:
future_dates = pd.DataFrame({'ds': test['Date']})

forecast_revenue = prophet_revenue.predict(future_dates)
revenue_predictions = forecast_revenue[['ds', 'yhat']].copy()
revenue_predictions.columns = ['Date', 'Revenue']


forecast_cogs = prophet_cogs.predict(future_dates)
cogs_predictions = forecast_cogs[['ds', 'yhat']].copy()
cogs_predictions.columns = ['Date', 'COGS']
    


## Blend tự chọn trọng số

In [ ]:
rev_blend = (
    0.4 * pd.Series(pred_revenue_gbm).reset_index(drop=True)
    + 0.3 * pd.Series(pred_revenue_xgb).reset_index(drop=True)
    + 0.3 * forecast_revenue['yhat'].reset_index(drop=True)
)

cogs_blend = (
    0.4 * pd.Series(pred_cogs_gbm).reset_index(drop=True)
    + 0.3 * pd.Series(pred_cogs_xgb).reset_index(drop=True)
    + 0.3 * forecast_cogs['yhat'].reset_index(drop=True)
)

submission_v1_blend = dfs['sample_submission'].copy()
submission_v1_blend['Date'] = pd.to_datetime(submission_v1_blend['Date']).dt.strftime('%Y-%m-%d')
submission_v1_blend['Revenue'] = rev_blend.clip(lower=0).values
submission_v1_blend['COGS'] = cogs_blend.clip(lower=0).values

submission_v1_blend.to_csv('submission_v1_blend.csv', index=False)
submission_v1_blend.head()


,Date,Revenue,COGS
0,2023-01-01,2.099300e+06,1.849897e+06
1,2023-01-02,1.549169e+06,1.344938e+06
2,2023-01-03,1.332714e+06,1.241428e+06
3,2023-01-04,1.554406e+06,1.468236e+06
4,2023-01-05,1.365505e+06,1.333001e+06


## Blend bằng cách đi tìm trọng số


In [ ]:
# 3-model blend search: gbm / xgboost / prophet only, averaged over multiple rolling windows
BLEND_STEP_3 = 0.05
N_BLEND_WINDOWS = 3

def build_blend_windows(df, horizon, n_windows=3):
    windows = []
    for offset in range(n_windows, 0, -1):
        start = len(df) - horizon * offset
        end = start + horizon
        if start <= 0:
            continue
        windows.append((start, end))
    return windows


def prophet_window_forecast(train_df, future_dates):
    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
        seasonality_mode='multiplicative',
        interval_width=0.95,
        changepoint_prior_scale=0.05,
    )
    model.fit(train_df)
    return model.predict(pd.DataFrame({'ds': future_dates}))['yhat'].values


def recursive_forecast_pair(rev_model, cogs_model, future_dates, history_frame):
    history_local = history_frame[['Date', 'Revenue', 'COGS']].copy().sort_values('Date').reset_index(drop=True)
    pred_rev = []
    pred_cogs = []

    for date in future_dates:
        X_input = create_features(date, history_local)
        X_rev = X_input.reindex(columns=x.columns, fill_value=0)
        X_cogs = X_input.reindex(columns=x1.columns, fill_value=0)

        rev_pred = rev_model.predict(X_rev)[0]
        cogs_pred = cogs_model.predict(X_cogs)[0]

        pred_rev.append(rev_pred)
        pred_cogs.append(cogs_pred)

        history_local = pd.concat([
            history_local,
            pd.DataFrame({
                'Date': [date],
                'Revenue': [rev_pred],
                'COGS': [cogs_pred],
            })
        ], ignore_index=True)

    return np.array(pred_rev), np.array(pred_cogs)


blend_windows = build_blend_windows(df, len(test), n_windows=N_BLEND_WINDOWS)
window_rev_frames = []
window_cogs_frames = []
window_actual_rev = []
window_actual_cogs = []

for window_id, (start, end) in enumerate(blend_windows, start=1):
    history_window = df[['Date', 'Revenue', 'COGS']].iloc[:start].copy()
    future_dates_window = df['Date'].iloc[start:end].reset_index(drop=True)
    actual_rev_window = df['Revenue'].iloc[start:end].reset_index(drop=True)
    actual_cogs_window = df['COGS'].iloc[start:end].reset_index(drop=True)

    bt_rev_gbm_w, bt_cogs_gbm_w = recursive_forecast_pair(gbm_rev_be, gbm_cogs_be, future_dates_window, history_window)
    bt_rev_xgb_w, bt_cogs_xgb_w = recursive_forecast_pair(xgb_rev_be, xgb_cogs_be, future_dates_window, history_window)

    revenue_window_df = revenue_df.iloc[:start].copy()
    cogs_window_df = cogs_df.iloc[:start].copy()
    bt_rev_prophet_w = prophet_window_forecast(revenue_window_df, future_dates_window)
    bt_cogs_prophet_w = prophet_window_forecast(cogs_window_df, future_dates_window)

    window_rev_frames.append(pd.DataFrame({
        'window': window_id,
        'gbm': bt_rev_gbm_w,
        'xgb': bt_rev_xgb_w,
        'prophet': bt_rev_prophet_w,
    }))
    window_cogs_frames.append(pd.DataFrame({
        'window': window_id,
        'gbm': bt_cogs_gbm_w,
        'xgb': bt_cogs_xgb_w,
        'prophet': bt_cogs_prophet_w,
    }))
    window_actual_rev.append(pd.Series(actual_rev_window.values))
    window_actual_cogs.append(pd.Series(actual_cogs_window.values))

rev_bt_3 = pd.concat(window_rev_frames, ignore_index=True)
cogs_bt_3 = pd.concat(window_cogs_frames, ignore_index=True)
backtest_actual_rev_multi = pd.concat(window_actual_rev, ignore_index=True)
backtest_actual_cogs_multi = pd.concat(window_actual_cogs, ignore_index=True)

rev_test_3 = pd.DataFrame({
    'gbm': pd.Series(pred_revenue_gbm).reset_index(drop=True),
    'xgb': pd.Series(pred_revenue_xgb).reset_index(drop=True),
    'prophet': forecast_revenue['yhat'].reset_index(drop=True),
})

cogs_test_3 = pd.DataFrame({
    'gbm': pd.Series(pred_cogs_gbm).reset_index(drop=True),
    'xgb': pd.Series(pred_cogs_xgb).reset_index(drop=True),
    'prophet': forecast_cogs['yhat'].reset_index(drop=True),
})

blend_results_3_rev = []
blend_results_3_cogs = []

for gbm_w in np.arange(0, 1.01, BLEND_STEP_3):
    for xgb_w in np.arange(0, 1.01 - gbm_w, BLEND_STEP_3):
        prophet_w = round(1.0 - gbm_w - xgb_w, 10)
        if prophet_w < 0:
            continue

        rev_bt_pred = gbm_w * rev_bt_3['gbm'] + xgb_w * rev_bt_3['xgb'] + prophet_w * rev_bt_3['prophet']
        cogs_bt_pred = gbm_w * cogs_bt_3['gbm'] + xgb_w * cogs_bt_3['xgb'] + prophet_w * cogs_bt_3['prophet']

        blend_results_3_rev.append({
            'gbm': gbm_w,
            'xgb': xgb_w,
            'prophet': prophet_w,
            'mae': metrics.mean_absolute_error(backtest_actual_rev_multi, rev_bt_pred),
        })

        blend_results_3_cogs.append({
            'gbm': gbm_w,
            'xgb': xgb_w,
            'prophet': prophet_w,
            'mae': metrics.mean_absolute_error(backtest_actual_cogs_multi, cogs_bt_pred),
        })

blend_results_3_rev = pd.DataFrame(blend_results_3_rev).sort_values('mae').reset_index(drop=True)
blend_results_3_cogs = pd.DataFrame(blend_results_3_cogs).sort_values('mae').reset_index(drop=True)

best_rev_3 = blend_results_3_rev.iloc[0]
best_cogs_3 = blend_results_3_cogs.iloc[0]

rev_pred_3 = (
    best_rev_3['gbm'] * rev_test_3['gbm']
    + best_rev_3['xgb'] * rev_test_3['xgb']
    + best_rev_3['prophet'] * rev_test_3['prophet']
)

cogs_pred_3 = (
    best_cogs_3['gbm'] * cogs_test_3['gbm']
    + best_cogs_3['xgb'] * cogs_test_3['xgb']
    + best_cogs_3['prophet'] * cogs_test_3['prophet']
)

print('Blend windows used:', blend_windows)
print('Best Revenue 3-model weights:')
print(best_rev_3)
display(blend_results_3_rev.head(15))

print('Best COGS 3-model weights:')
print(best_cogs_3)
display(blend_results_3_cogs.head(15))

submission_blend_search_3 = dfs['sample_submission'].copy()
submission_blend_search_3['Date'] = pd.to_datetime(submission_blend_search_3['Date']).dt.strftime('%Y-%m-%d')
submission_blend_search_3['Revenue'] = rev_pred_3.clip(lower=0).values
submission_blend_search_3['COGS'] = cogs_pred_3.clip(lower=0).values
submission_blend_search_3.to_csv('submission_v2 - blend_search.csv', index=False)
submission_blend_search_3.head()


13:18:16 - cmdstanpy - INFO - Chain [1] start processing
13:18:16 - cmdstanpy - INFO - Chain [1] done processing
13:18:16 - cmdstanpy - INFO - Chain [1] start processing
13:18:17 - cmdstanpy - INFO - Chain [1] done processing
13:18:21 - cmdstanpy - INFO - Chain [1] start processing
13:18:22 - cmdstanpy - INFO - Chain [1] done processing
13:18:22 - cmdstanpy - INFO - Chain [1] start processing
13:18:22 - cmdstanpy - INFO - Chain [1] done processing
13:18:27 - cmdstanpy - INFO - Chain [1] start processing
13:18:27 - cmdstanpy - INFO - Chain [1] done processing
13:18:27 - cmdstanpy - INFO - Chain [1] start processing
13:18:28 - cmdstanpy - INFO - Chain [1] done processing


Blend windows used: [(2189, 2737), (2737, 3285), (3285, 3833)]
Best Revenue 3-model weights:
gbm        5.500000e-01
xgb        0.000000e+00
prophet    4.500000e-01
mae        1.131441e+06
Name: 0, dtype: float64


,gbm,xgb,prophet,mae
0,0.55,0.00,0.45,1.131441e+06
1,0.50,0.00,0.50,1.132035e+06
2,0.50,0.05,0.45,1.133255e+06
3,0.45,0.05,0.50,1.133415e+06
4,0.40,0.10,0.50,1.135535e+06
5,0.45,0.10,0.45,1.135965e+06
6,0.35,0.15,0.50,1.138449e+06
7,0.40,0.15,0.45,1.139444e+06
8,0.30,0.20,0.50,1.142200e+06
9,0.35,0.20,0.45,1.143503e+06


Best COGS 3-model weights:
gbm             0.050000
xgb             0.400000
prophet         0.550000
mae        965102.556384
Name: 0, dtype: float64


,gbm,xgb,prophet,mae
0,0.05,0.40,0.55,965102.556384
1,0.00,0.45,0.55,965243.665764
2,0.10,0.35,0.55,965257.194568
3,0.15,0.30,0.55,965572.032449
4,0.20,0.25,0.55,966275.879733
5,0.25,0.20,0.55,967260.927931
6,0.30,0.15,0.55,968476.127045
7,0.35,0.10,0.55,969910.533263
8,0.40,0.05,0.55,971598.838522
9,0.10,0.30,0.60,972055.785597


,Date,Revenue,COGS
0,2023-01-01,1.988102e+06,1.920008e+06
1,2023-01-02,1.537989e+06,1.604560e+06
2,2023-01-03,1.547896e+06,1.522285e+06
3,2023-01-04,1.682386e+06,1.710675e+06
4,2023-01-05,1.486067e+06,1.560094e+06
